# 02 · 그래프와 학습 입력 준비

01 완료 후 실행합니다. 전역 CSR 인덱스와 전체 대상 목록은 CPU에서 만들고,
반복 학습에서는 **CUDA로 2홉 탐색·중복 제거·공유 그래프·노드 매핑**을 처리합니다.
2홉, 방향별 다른 분 거래 최대 100개 + 같은 분 거래 전부, 고정 우선순위, 일별 마감 cutoff를 유지합니다.

거래 입력은 수치 float32 + 범주 uint8로 **거래 ID당 49바이트**를 GPU에 보관하고,
필요할 때 24차원으로 조합합니다. 일별 계좌 피처와 원래 degree만 날짜마다 갱신합니다.
간선 6,000,000개를 넘는 배치는 피처 조합 전에 재귀 분할하며, 단일 거래 그래프는 자르거나 누락하지 않습니다.
GPU 상주 피처의 미래 날짜 항목은 샘플러의 cutoff로 선택을 막습니다. 라벨은 캐시에 포함하지 않습니다.

`validate`는 선택한 장치에서 CPU 기준 그래프·입력과 비교하고 4개 모델의 forward/backward를 검사합니다.
CPU 검증만으로 CUDA 실행을 시작할 수 없습니다. 전체 대상 학습·평가 결과를 검증한 단계는 아닙니다.


In [ ]:
from pathlib import Path
import json
import sys
import subprocess

PROJECT_DIR = Path.cwd().resolve()
link = PROJECT_DIR / ".salt_project.json"
if not link.exists():
    raise FileNotFoundError("이 노트북들이 있는 폴더에서 00_환경설정과_실행준비.ipynb를 먼저 실행하세요.")
project = json.loads(link.read_text())
WORK_DIR = Path(project["work_dir"])
DEVICE = project["device"]
for module_name in ("salt_daily", "runtime", "run", "notebook_control"):
    loaded = sys.modules.get(module_name)
    location = getattr(loaded, "__file__", None)
    if location and not Path(location).resolve().is_relative_to(WORK_DIR):
        raise RuntimeError("다른 WORK_DIR가 import되어 있습니다. 커널을 재시작하세요.")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

def command(stage, device=DEVICE):
    subprocess.run([sys.executable, "-u", str(WORK_DIR/"run.py"), stage, "--device", device], cwd=WORK_DIR, check=True)

print("작업 폴더:", WORK_DIR)


### 이 단계의 실행 코드

아래 소스는 00 노트북이 읽어 작업 폴더에 실행 파일로 만듭니다. 코드를 수정하면 노트북을 저장하고 **새 WORK_DIR**로 00을 다시 실행하세요.

In [ ]:
SOURCES = {}


In [ ]:
# salt_daily/graph_prepare.py
SOURCES['salt_daily/graph_prepare.py'] = '''"""Global CSR indices and full supervised populations from fresh raw data."""
from pathlib import Path
import hashlib
import json
import numpy as np
from .config import ROOT,DATA_ROOT
from .storage import read_json,write_json,sha,utc
from .preprocessing import stage,config
from .raw_common import export_array
from .population import build_population,EXPECTED


def build_graph():
    with stage('04_graph',('01_ingest','02_edges','03_daily_nodes')) as c:
        if c is None:return
        cfg=config();meta=read_json(DATA_ROOT/'raw_metadata.json');n=meta['rows'];nn=meta['nodes']
        folder=DATA_ROOT/'graph/hi';folder.mkdir(parents=True,exist_ok=True)
        # Explicit integer casts preserve the existing DuckDB hash argument types.
        c.execute('CREATE TEMP VIEW graph_raw AS SELECT edge_id::INTEGER AS eid,src::INTEGER AS src,dst::INTEGER AS dst,minute::INTEGER AS minute FROM features')
        for role,column in [('incoming','dst'),('outgoing','src')]:
            for kind,order in [('priority',f'hash(eid,{column},42),eid'),('time','minute,eid')]:
                degree=np.zeros(nn,np.int64);cursor=0
                ids=np.lib.format.open_memmap(folder/f'{role}_{kind}_ids.npy','w+','uint32',(n,))
                for batch in c.execute(f'SELECT {column} node,eid FROM graph_raw ORDER BY {column},{order}').to_arrow_reader(cfg.chunk_rows):
                    nodes=batch.column(0).to_numpy();values=batch.column(1).to_numpy()
                    ids[cursor:cursor+len(values)]=values;degree+=np.bincount(nodes,minlength=nn);cursor+=len(values)
                assert cursor==n and degree.sum()==n
                ids.flush();del ids
                np.save(folder/f'{role}_{kind}_offsets.npy',np.r_[0,np.cumsum(degree)])
                print('graph_index',role,kind,'rows',n,flush=True)
        write_json(folder/'complete.json',dict(rows=n,nodes=nn,width=meta['width'],version=3,complete=True))


def prepare_inputs():
    with stage('05_inputs',('01_ingest','02_edges','03_daily_nodes','04_graph')) as c:
        if c is None:return read_json(DATA_ROOT/'manifest.json')
        population=build_population(ROOT/'cache/population',
            np.load(DATA_ROOT/'features/minute.npy',mmap_mode='r'),np.load(DATA_ROOT/'features/present.npy',mmap_mode='r'),
            np.load(DATA_ROOT/'labels/assets/positive_ids.npy',mmap_mode='r'),np.load(DATA_ROOT/'labels/assets/positive_classes.npy',mmap_mode='r'))
        counts={k:v['rows'] for k,v in population['cohorts'].items()}
        if config().require_hi_large_counts and counts!=EXPECTED:
            raise ValueError(f'Unexpected HI-Large population: {counts}; check the supplied raw files')
        files=[]
        for group in ('features','structural','ports','labels','graph'):
            for path in sorted((DATA_ROOT/group).rglob('*')):
                if not path.is_file() or path.suffix not in ('.npy','.json'):continue
                s=path.stat();item=dict(path=str(path.relative_to(DATA_ROOT)),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
                if path.suffix=='.npy':
                    a=np.load(path,mmap_mode='r');item.update(dtype=a.dtype.str,shape=list(a.shape));del a
                files.append(item)
        manifest=dict(schema='salt-daily-raw-v1',raw=read_json(DATA_ROOT/'raw_metadata.json'),files=files,
            cohorts=population['cohorts'],population_manifest_sha256=sha(ROOT/'cache/population/population.json'),
            calibration=read_json(DATA_ROOT/'calibration.json'),calibration_sha256=sha(DATA_ROOT/'calibration.json'),
            preprocessing_identity=read_json(DATA_ROOT/'_build/01_ingest.json')['identity'],graph_code_sha256=sha(Path(__file__)))
        manifest['dataset_id']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
        write_json(DATA_ROOT/'manifest.json',manifest)
        protocol=read_json(ROOT/'protocol.template.json')
        protocol['cohorts']=population['cohorts'];protocol['population_manifest_sha256']=manifest['population_manifest_sha256']
        protocol['dataset_id']=manifest['dataset_id'];protocol['input_manifest_sha256']=sha(DATA_ROOT/'manifest.json')
        protocol['training']['targets_per_epoch']=counts['train_gnn']
        protocol['training']['full_train_population']=counts['train_gnn']
        protocol['training']['targets_per_six_epochs']=counts['train_gnn']*protocol['training']['epochs_primary']
        protocol['training']['updates_per_full_epoch']=sum((s['stop']-s['start']+protocol['training']['logical_batch']-1)//protocol['training']['logical_batch'] for s in population['cohorts']['train_gnn']['spans'])
        protocol['evaluation']['full_validation_rows']=counts['validation'];protocol['evaluation']['full_historical_test_rows']=counts['test']
        if (ROOT/'runs/daily_close_001/models').exists():raise RuntimeError('Use a fresh WORK_DIR for a new dataset')
        write_json(ROOT/'protocol.json',protocol)
        write_json(ROOT/'calibration.json',manifest['calibration'])
        return manifest
'''


In [ ]:
# salt_daily/population.py
SOURCES['salt_daily/population.py'] = '''"""All observed targets, indexed by day on disk. Labels never select targets."""
import fcntl
from pathlib import Path

import numpy as np

from .config import ROOT, FEATURES, FRAUDGT
from .storage import read_json, write_json, sha, utc

WINDOWS = {'train_gnn': (7, 55), 'monitor': (56, 64),
           'validation': (69, 73), 'test': (74, 91)}
EXPECTED = {'train_gnn': 87572226, 'monitor': 18625135,
            'validation': 8551015, 'test': 33429507}


def labels_for(ids, positive_ids, positive_classes):
    ids = np.asarray(ids)
    positions = np.searchsorted(positive_ids, ids)
    found = np.flatnonzero(positions < len(positive_ids))
    found = found[positive_ids[positions[found]] == ids[found]]
    labels = np.zeros(len(ids), np.int8)
    labels[found] = positive_classes[positions[found]]
    return labels


def build_population(directory, minute, present, positive_ids, positive_classes,
                     windows=WINDOWS, block=1_000_000):
    """Two bounded-memory scans; each retained ID appears once in its date range."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    assert len(minute) == len(present) and len(minute) < 2**32
    assert np.all(positive_ids[1:] > positive_ids[:-1])
    assert np.all((positive_classes >= 1) & (positive_classes <= 9))
    ndays = max(end for _, end in windows.values()) + 1
    counts = np.zeros(ndays, np.int64)
    for start in range(0, len(minute), block):
        days = minute[start:start+block] // 1440
        valid = (present[start:start+len(days)] != 0) & (days >= 0) & (days < ndays)
        counts += np.bincount(days[valid], minlength=ndays)
    outputs, metadata, cursor = {}, {}, {}
    for scope, (first, last) in windows.items():
        boundaries = np.r_[0, np.cumsum(counts[first:last+1])]
        n = int(boundaries[-1])
        assert n > 0
        outputs[scope] = (
            np.lib.format.open_memmap(directory/f'{scope}_ids.npy', mode='w+', dtype='uint32', shape=(n,)),
            np.lib.format.open_memmap(directory/f'{scope}_labels.npy', mode='w+', dtype='int8', shape=(n,)))
        metadata[scope] = dict(rows=n, min_day=first, max_day=last, positive=0, unmatched=0,
                               spans=[dict(day=day, start=int(boundaries[day-first]),
                                           stop=int(boundaries[day-first+1])) for day in range(first, last+1)])
        cursor[scope] = boundaries[:-1].copy()
    for start in range(0, len(minute), block):
        days = minute[start:start+block] // 1440
        valid = present[start:start+len(days)] != 0
        for scope, (first, last) in windows.items():
            ids = (np.flatnonzero(valid & (days >= first) & (days <= last)) + start).astype(np.uint32)
            if not len(ids):
                continue
            labels = labels_for(ids, positive_ids, positive_classes)
            metadata[scope]['positive'] += int(np.count_nonzero(labels))
            metadata[scope]['unmatched'] += int(np.count_nonzero(labels == 9))
            selected_days = minute[ids] // 1440
            for day in np.unique(selected_days):
                take = selected_days == day
                n = int(take.sum()); offset = int(cursor[scope][day-first])
                outputs[scope][0][offset:offset+n] = ids[take]
                outputs[scope][1][offset:offset+n] = labels[take]
                cursor[scope][day-first] += n
    for scope, item in metadata.items():
        first, last = windows[scope]
        assert np.array_equal(cursor[scope], [span['stop'] for span in item['spans']])
        eligible = ((minute[positive_ids] >= first*1440) &
                    (minute[positive_ids] < (last+1)*1440) & (present[positive_ids] != 0))
        assert item['positive'] == int(eligible.sum())
        assert item['unmatched'] == int(np.count_nonzero(eligible & (positive_classes == 9)))
        item['normal'] = item['rows'] - item['positive']
        for values in outputs[scope]:
            values.flush()
        ids = outputs[scope][0]
        # Ordered, unique IDs per day + matching day counts prove full membership.
        for span in item['spans']:
            values = ids[span['start']:span['stop']]
            assert np.all(values[1:] > values[:-1])
            assert np.all(minute[values] // 1440 == span['day'])
            assert np.all(present[values] != 0)
        item['files'] = {f'{scope}_{suffix}.npy': sha(directory/f'{scope}_{suffix}.npy')
                         for suffix in ('ids', 'labels')}
    result = dict(created_utc=utc(), cohorts=metadata, daily_counts=counts.tolist(),
                  selection='observed-field dedup present mask and date only',
                  target_downsampling=False, ipw=False, windows=windows,
                  every_eligible_target_once=True)
    write_json(directory/'population.json', result)
    return result


def prepare_population():
    directory = ROOT/'cache/population'
    directory.mkdir(parents=True, exist_ok=True)
    with (directory/'prepare.lock').open('a') as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        marker = directory/'population.json'
        if marker.exists():
            result = read_json(marker)
        else:
            result = build_population(directory,
                np.load(FEATURES/'minute.npy', mmap_mode='r'),
                np.load(FEATURES/'present.npy', mmap_mode='r'),
                np.load(FRAUDGT/'assets/positive_ids.npy', mmap_mode='r'),
                np.load(FRAUDGT/'assets/positive_classes.npy', mmap_mode='r'))
        for scope, expected in EXPECTED.items():
            assert result['cohorts'][scope]['rows'] == expected, scope
        return result


class FullPopulation:
    def __init__(self, scope, directory=None):
        directory = Path(directory) if directory else ROOT/'cache/population'
        self.meta = read_json(directory/'population.json')['cohorts'][scope]
        self.scope = scope
        self.ids = np.load(directory/f'{scope}_ids.npy', mmap_mode='r')
        self.y = np.load(directory/f'{scope}_labels.npy', mmap_mode='r')
        assert len(self.ids) == len(self.y) == self.meta['rows']

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, key):
        # Deliberately no sample-weight field and no fallback to legacy cohorts.
        return {'ids': self.ids, 'y': self.y}[key]

    def orders(self, epoch, shuffle=True, start_day_cursor=0):
        for span in self.meta['spans'][start_day_cursor:]:
            order = np.arange(span['start'], span['stop'], dtype=np.int64)
            if shuffle:
                np.random.default_rng(42000 + epoch*100 + span['day']).shuffle(order)
            yield span['day'], order

    def updates(self, batch_size):
        return sum((s['stop']-s['start']+batch_size-1)//batch_size for s in self.meta['spans'])
'''


In [ ]:
# salt_daily/prepare.py
SOURCES['salt_daily/prepare.py'] = '''"""Ensure the fresh raw preprocessing and graph stages are complete."""
from .config import ROOT, DATA_ROOT
from .storage import read_json


def prepare():
    from .portable import check_inputs
    check_inputs()
    return read_json(ROOT/'protocol.json')
'''


In [ ]:
# runtime/sampler.c
SOURCES['runtime/sampler.c'] = '''#include <stdint.h>
#include <stdlib.h>
#include <string.h>
typedef int64_t I;
typedef struct {I *p; size_t n,cap;} Vec;
typedef struct {I id,flag;} Edge;
typedef struct {Edge *p; size_t n,cap;} Edges;
typedef struct {Vec a[8];} Result;
static int push(Vec*v,I x){if(v->n==v->cap){size_t cap=v->cap?v->cap*2:64;if(cap>SIZE_MAX/sizeof(I))return 0;void*p=realloc(v->p,cap*sizeof(I));if(!p)return 0;v->p=p;v->cap=cap;}v->p[v->n++]=x;return 1;}
static int epush(Edges*v,I id,I flag){if(v->n==v->cap){size_t cap=v->cap?v->cap*2:64;if(cap>SIZE_MAX/sizeof(Edge))return 0;void*p=realloc(v->p,cap*sizeof(Edge));if(!p)return 0;v->p=p;v->cap=cap;}v->p[v->n++]=(Edge){id,flag};return 1;}
static int cmpi(const void*a,const void*b){I x=*(const I*)a,y=*(const I*)b;return(x>y)-(x<y);}
static int cmpe(const void*a,const void*b){I x=((const Edge*)a)->id,y=((const Edge*)b)->id;return(x>y)-(x<y);}
static void unique(Vec*v){if(v->n<2)return;qsort(v->p,v->n,sizeof(I),cmpi);size_t j=1;for(size_t k=1;k<v->n;k++)if(v->p[k]!=v->p[j-1])v->p[j++]=v->p[k];v->n=j;}
static void merge(Edges*v){if(v->n<2)return;qsort(v->p,v->n,sizeof(Edge),cmpe);size_t j=1;for(size_t k=1;k<v->n;k++){if(v->p[k].id==v->p[j-1].id)v->p[j-1].flag|=v->p[k].flag;else v->p[j++]=v->p[k];}v->n=j;}
static I find(Vec*v,I x){size_t lo=0,hi=v->n;while(lo<hi){size_t m=lo+(hi-lo)/2;if(v->p[m]<x)lo=m+1;else hi=m;}return (I)lo;}
static I lower(const uint32_t*ids,const int32_t*minute,I lo,I hi,I target){while(lo<hi){I m=lo+(hi-lo)/2;if(minute[ids[m]]<target)lo=m+1;else hi=m;}return lo;}
static int select_edges(Vec*nodes,Edges*edges,const int32_t*minute,I target,I end,I cap,I nn,
 const I*ipoff,const uint32_t*ip,const I*itoff,const uint32_t*it,
 const I*opoff,const uint32_t*op,const I*otoff,const uint32_t*ot){
 for(int dir=0;dir<2;dir++){
  const I*poff=dir?opoff:ipoff;const uint32_t*pi=dir?op:ip;
  const I*toff=dir?otoff:itoff;const uint32_t*ti=dir?ot:it;I flag=dir?2:1;
  for(size_t k=0;k<nodes->n;k++){
   I node=nodes->p[k];if(node<0||node>=nn)return 0;I found=0;
   for(I j=poff[node];j<poff[node+1]&&found<cap;j++){
    I id=pi[j];if(minute[id]>=end||minute[id]==target)continue;
    if(!epush(edges,id,flag))return 0;found++;
   }
   I lo=lower(ti,minute,toff[node],toff[node+1],target),hi=lower(ti,minute,lo,toff[node+1],target+1);
   for(I j=lo;j<hi;j++)if(!epush(edges,ti[j],flag))return 0;
  }
 }return 1;
}
void free_result(Result*r){if(r){for(int k=0;k<8;k++)free(r->a[k].p);free(r);}}
I result_size(Result*r,int k){return (I)r->a[k].n;}
I*result_data(Result*r,int k){return r->a[k].p;}
Result*sample_batch(const int32_t*src,const int32_t*dst,const int32_t*minute,I width,const I*targets,I count,I nn,I cap,
 const I*ipoff,const uint32_t*ip,const I*itoff,const uint32_t*it,
 const I*opoff,const uint32_t*op,const I*otoff,const uint32_t*ot){
 Result*r=calloc(1,sizeof(Result));Vec roots={0},frontier={0},nodes={0};Edges edges={0};I*node_map=NULL;if(!r)return NULL;
 node_map=malloc((size_t)nn*sizeof(I));if(!node_map)goto fail;
 for(I t=0;t<count;t++){
  I id=targets[t];if(id<0||id>=width)goto fail;I s=src[id],d=dst[id],tm=minute[id],end=(tm/1440+1)*1440;
  roots.n=frontier.n=nodes.n=edges.n=0;if(!push(&roots,s)||!push(&roots,d))goto fail;unique(&roots);
  if(!select_edges(&roots,&edges,minute,tm,end,cap,nn,ipoff,ip,itoff,it,opoff,op,otoff,ot))goto fail;merge(&edges);
  for(size_t k=0;k<edges.n;k++){I e=edges.p[k].id,f=edges.p[k].flag;if((f&1)&&!push(&frontier,src[e]))goto fail;if((f&2)&&!push(&frontier,dst[e]))goto fail;}
  unique(&frontier);
  if(!select_edges(&frontier,&edges,minute,tm,end,cap,nn,ipoff,ip,itoff,it,opoff,op,otoff,ot))goto fail;merge(&edges);
  if(!push(&nodes,s)||!push(&nodes,d))goto fail;
  for(size_t k=0;k<edges.n;k++){I e=edges.p[k].id;if(!push(&nodes,src[e])||!push(&nodes,dst[e]))goto fail;}unique(&nodes);
  I offset=(I)r->a[0].n;
  for(size_t k=0;k<nodes.n;k++){I n=nodes.p[k];if(n<0||n>=nn)goto fail;node_map[n]=offset+(I)k;if(!push(&r->a[0],n)||!push(&r->a[7],n==s||n==d))goto fail;}
  for(size_t k=0;k<edges.n;k++){I e=edges.p[k].id;if(!push(&r->a[1],e)||!push(&r->a[2],node_map[src[e]])||!push(&r->a[3],node_map[dst[e]])||!push(&r->a[6],edges.p[k].flag))goto fail;}
  if(!push(&r->a[4],node_map[s])||!push(&r->a[5],node_map[d]))goto fail;
 }
 free(node_map);free(roots.p);free(frontier.p);free(nodes.p);free(edges.p);return r;
 fail:free(node_map);free(roots.p);free(frontier.p);free(nodes.p);free(edges.p);free_result(r);return NULL;
}
'''


In [ ]:
# runtime/graph_runtime.py
SOURCES['runtime/graph_runtime.py'] = '''"""Exact HI-Large input assembly with relocatable paths and owned native copies."""
from pathlib import Path
import json
import ctypes as C
import subprocess
import numpy as np
from salt_daily.config import ROOT, GRAPH, STRUCTURAL, FEATURES as G
from salt_daily.storage import sha
CACHE = GRAPH
P = ROOT/'runtime'

def load_reference_native():
    source = P/'sampler.c'
    cache = ROOT/'cache/native'
    cache.mkdir(parents=True, exist_ok=True)
    so = cache/('sampler_'+sha(source)[:16]+'.so')
    if not so.exists():
        subprocess.run(['gcc','-O3','-std=c11','-fPIC','-shared',str(source),'-o',str(so)],check=True,capture_output=True)
    lib=C.CDLL(str(so)); i=C.c_int64; p=C.POINTER(i); p32=C.POINTER(C.c_int32); u32=C.POINTER(C.c_uint32)
    lib.sample_batch.argtypes=[p32,p32,p32,i,p,i,i,i]+[p,u32]*4;lib.sample_batch.restype=C.c_void_p
    lib.result_size.argtypes=[C.c_void_p,C.c_int];lib.result_size.restype=i
    lib.result_data.argtypes=[C.c_void_p,C.c_int];lib.result_data.restype=p
    lib.free_result.argtypes=[C.c_void_p];lib.free_result.restype=None
    return lib


class Topology:
    def __init__(self,dataset,cap=100):
        folder=CACHE/dataset;meta=json.loads((folder/'complete.json').read_text());self.nn=meta['nodes'];self.width=meta['width'];self.cap=cap
        raw=G if dataset=='hi' else folder
        self.src=np.load(raw/'src.npy',mmap_mode='r');self.dst=np.load(raw/'dst.npy',mmap_mode='r');self.minute=np.load(raw/'minute.npy',mmap_mode='r')
        lib=load_reference_native();self.lib=lib
        p=C.POINTER(C.c_int64);p32=C.POINTER(C.c_int32);u32=C.POINTER(C.c_uint32)
        self.rawptr=[a.ctypes.data_as(p32) for a in [self.src,self.dst,self.minute]]
        self.indices=[];self.indexptr=[]
        for role in ['incoming','outgoing']:
            for kind in ['priority','time']:
                for name,typ in [('offsets',p),('ids',u32)]:
                    a=np.load(folder/(role+'_'+kind+'_'+name+'.npy'),mmap_mode='r');self.indices.append(a);self.indexptr.append(a.ctypes.data_as(typ))
    def batch(self,ids):
        ids=np.ascontiguousarray(ids,dtype=np.int64);assert len(ids)>0 and ids.min()>=0 and ids.max()<self.width
        handle=self.lib.sample_batch(*self.rawptr,self.width,ids.ctypes.data_as(C.POINTER(C.c_int64)),len(ids),self.nn,self.cap,*self.indexptr)
        if not handle:raise RuntimeError('Native daily graph construction failed')
        try:
            from salt_daily.shared_topology import copy_result
            return tuple(copy_result(self.lib,handle,k) for k in range(8))
        finally:self.lib.free_result(handle)

class HIInputs:
    def __init__(self):
        self.edge=np.load(STRUCTURAL/'x.npy',mmap_mode='r')
        self.present=np.load(STRUCTURAL/'present.npy',mmap_mode='r')
        self.z=np.load(G/'z.npy',mmap_mode='r');self.pz=np.load(CACHE/'hi/paid_z.npy',mmap_mode='r')
        self.minute=np.load(G/'minute.npy',mmap_mode='r')
        self.test=None;self.day=None
    def set_day(self,date):
        self.day=date;self.nodes=np.load(G/'nodes'/(date+'.npy'),mmap_mode='r')
    def edges(self,ids):
        ids=np.asarray(ids,np.int64);a=np.empty((len(ids),17),np.float32);ok=ids<len(self.edge)
        if ok.any():ok[ok]=self.present[ids[ok]]!=0
        if ok.any():a[ok]=self.edge[ids[ok]]
        if (~ok).any():
            raise ValueError('Missing structural features in fresh raw cache')
        a[:,7]=self.pz[ids];a[:,16]=self.z[ids];return a
    def materialize(self,ids,topology):
        n,e,s,d,ts,td,flags,ego=topology
        x=np.column_stack([self.nodes[n],ego.astype(np.float32)])
        return x,np.stack([s,d]),self.edges(e),np.stack([ts,td]),self.edges(ids),(flags&1)>0,(flags&2)>0
'''


In [ ]:
# salt_daily/shared_sampler.c
SOURCES['salt_daily/shared_sampler.c'] = '''#include <stdint.h>
#include <stdlib.h>
#include <string.h>
typedef int64_t I;
typedef struct {I *p; size_t n,cap;} Vec;
typedef struct {I id,flag;} Edge;
typedef struct {Edge *p; size_t n,cap;} Edges;
typedef struct {Vec a[8];} Result;
static int push(Vec*v,I x){if(v->n==v->cap){size_t cap=v->cap?v->cap*2:64;if(cap>SIZE_MAX/sizeof(I))return 0;void*p=realloc(v->p,cap*sizeof(I));if(!p)return 0;v->p=p;v->cap=cap;}v->p[v->n++]=x;return 1;}
static int epush(Edges*v,I id,I flag){if(v->n==v->cap){size_t cap=v->cap?v->cap*2:64;if(cap>SIZE_MAX/sizeof(Edge))return 0;void*p=realloc(v->p,cap*sizeof(Edge));if(!p)return 0;v->p=p;v->cap=cap;}v->p[v->n++]=(Edge){id,flag};return 1;}
static int cmpi(const void*a,const void*b){I x=*(const I*)a,y=*(const I*)b;return(x>y)-(x<y);}
static int cmpe(const void*a,const void*b){I x=((const Edge*)a)->id,y=((const Edge*)b)->id;return(x>y)-(x<y);}
static void unique(Vec*v){if(v->n<2)return;qsort(v->p,v->n,sizeof(I),cmpi);size_t j=1;for(size_t k=1;k<v->n;k++)if(v->p[k]!=v->p[j-1])v->p[j++]=v->p[k];v->n=j;}
static void merge(Edges*v){if(v->n<2)return;qsort(v->p,v->n,sizeof(Edge),cmpe);size_t j=1;for(size_t k=1;k<v->n;k++){if(v->p[k].id==v->p[j-1].id)v->p[j-1].flag|=v->p[k].flag;else v->p[j++]=v->p[k];}v->n=j;}
static I find(Vec*v,I x){size_t lo=0,hi=v->n;while(lo<hi){size_t m=lo+(hi-lo)/2;if(v->p[m]<x)lo=m+1;else hi=m;}return (I)lo;}
static I lower(const uint32_t*ids,const int32_t*minute,I lo,I hi,I target){while(lo<hi){I m=lo+(hi-lo)/2;if(minute[ids[m]]<target)lo=m+1;else hi=m;}return lo;}
/* A state packs account and target-minute-within-day. Multiple query
 * identities with the same state have exactly the same outgoing expansion. */
static int add_selected(Edges*edges,Vec*next,const int32_t*src,const int32_t*dst,I edge,int dir,I minute_offset){
 if(!epush(edges,edge,dir?2:1))return 0;
 if(next){I neighbor=dir?dst[edge]:src[edge];if(!push(next,neighbor*1440+minute_offset))return 0;}
 return 1;
}
static int expand_states(Vec*states,Edges*edges,Vec*next,const int32_t*src,const int32_t*dst,const int32_t*minute,I day,I cap,I nn,
 const I*ipoff,const uint32_t*ip,const I*itoff,const uint32_t*it,
 const I*opoff,const uint32_t*op,const I*otoff,const uint32_t*ot){
 I end=(day+1)*1440;
 for(size_t k=0;k<states->n;k++){
  I node=states->p[k]/1440,offset=states->p[k]%1440,target=day*1440+offset;
  if(node<0||node>=nn)return 0;
  for(int dir=0;dir<2;dir++){
   const I*poff=dir?opoff:ipoff;const uint32_t*pi=dir?op:ip;
   const I*toff=dir?otoff:itoff;const uint32_t*ti=dir?ot:it;
   I found=0;
   for(I j=poff[node];j<poff[node+1]&&found<cap;j++){
    I id=pi[j];if(minute[id]>=end||minute[id]==target)continue;
    if(!add_selected(edges,next,src,dst,id,dir,offset))return 0;found++;
   }
   I lo=lower(ti,minute,toff[node],toff[node+1],target),hi=lower(ti,minute,lo,toff[node+1],target+1);
   for(I j=lo;j<hi;j++)if(!add_selected(edges,next,src,dst,ti[j],dir,offset))return 0;
  }
 }
 return 1;
}
void free_result(Result*r){if(r){for(int k=0;k<8;k++)free(r->a[k].p);free(r);}}
I result_size(Result*r,int k){return (I)r->a[k].n;}
I*result_data(Result*r,int k){return r->a[k].p;}
Result*sample_batch(const int32_t*src,const int32_t*dst,const int32_t*minute,I width,const I*targets,I count,I nn,I cap,
 const I*ipoff,const uint32_t*ip,const I*itoff,const uint32_t*it,
 const I*opoff,const uint32_t*op,const I*otoff,const uint32_t*ot){
 Result*r=calloc(1,sizeof(Result));Vec roots={0},frontier={0},nodes={0};Edges edges={0};I*node_map=NULL;if(!r)return NULL;
 if(count<=0)goto fail;
 I day=-1;
 for(I t=0;t<count;t++){
  I id=targets[t];if(id<0||id>=width)goto fail;I s=src[id],d=dst[id],tm=minute[id],current_day=tm/1440;
  if(s<0||s>=nn||d<0||d>=nn||tm<0)goto fail;
  if(t==0)day=current_day;else if(current_day!=day)goto fail;
  if(!push(&roots,s*1440+tm%1440)||!push(&roots,d*1440+tm%1440))goto fail;
  if(!push(&nodes,s)||!push(&nodes,d))goto fail;
 }
 unique(&roots);
 if(!expand_states(&roots,&edges,&frontier,src,dst,minute,day,cap,nn,ipoff,ip,itoff,it,opoff,op,otoff,ot))goto fail;
 unique(&frontier);
 if(!expand_states(&frontier,&edges,NULL,src,dst,minute,day,cap,nn,ipoff,ip,itoff,it,opoff,op,otoff,ot))goto fail;
 merge(&edges);
 for(size_t k=0;k<edges.n;k++){I e=edges.p[k].id;if(!push(&nodes,src[e])||!push(&nodes,dst[e]))goto fail;}
 unique(&nodes);node_map=malloc((size_t)nn*sizeof(I));if(!node_map)goto fail;
 for(size_t k=0;k<nodes.n;k++){I n=nodes.p[k];if(n<0||n>=nn)goto fail;node_map[n]=(I)k;if(!push(&r->a[0],n)||!push(&r->a[7],0))goto fail;}
 for(size_t k=0;k<edges.n;k++){I e=edges.p[k].id;if(!push(&r->a[1],e)||!push(&r->a[2],node_map[src[e]])||!push(&r->a[3],node_map[dst[e]])||!push(&r->a[6],edges.p[k].flag))goto fail;}
 for(I t=0;t<count;t++){
  I id=targets[t],s=node_map[src[id]],d=node_map[dst[id]];
  if(!push(&r->a[4],s)||!push(&r->a[5],d))goto fail;r->a[7].p[s]=r->a[7].p[d]=1;
 }
 free(node_map);free(roots.p);free(frontier.p);free(nodes.p);free(edges.p);return r;
 fail:free(node_map);free(roots.p);free(frontier.p);free(nodes.p);free(edges.p);free_result(r);return NULL;
}
'''


In [ ]:
# salt_daily/shared_topology.py
SOURCES['salt_daily/shared_topology.py'] = '''"""Batch-wide account/minute expansion, equivalent to the prior shared union.

Accounts with different query minutes remain distinct sampling states so the
same-minute inclusion rule is preserved. Final nodes and transaction IDs share
one graph with exactly the prior target order, direction flags and batch Ego.
"""
import ctypes as C
import subprocess
import numpy as np
from .config import ROOT
from .storage import sha
from runtime.graph_runtime import Topology


def copy_result(lib, handle, index):
    """Own the result without caching a ctypes array type for every length."""
    size = lib.result_size(handle, index)
    if size < 0:
        raise RuntimeError('Invalid native result size')
    result = np.empty(size, dtype=np.int64)
    if size:
        pointer = lib.result_data(handle, index)
        if not pointer:
            raise RuntimeError('Missing native result data')
        C.memmove(result.ctypes.data, pointer, result.nbytes)
    return result


def load_native():
    source = ROOT/'salt_daily/shared_sampler.c'
    directory = ROOT/'cache/shared_sampler'
    directory.mkdir(parents=True, exist_ok=True)
    path = directory/f'shared_{sha(source)[:16]}.so'
    if not path.exists():
        temporary = path.with_suffix('.tmp.so')
        subprocess.run(['gcc','-O3','-std=c11','-fPIC','-shared',str(source),'-o',str(temporary)],
                       check=True, capture_output=True)
        temporary.replace(path)
    lib = C.CDLL(str(path))
    i=C.c_int64; p=C.POINTER(i); p32=C.POINTER(C.c_int32); u32=C.POINTER(C.c_uint32)
    lib.sample_batch.argtypes=[p32,p32,p32,i,p,i,i,i]+[p,u32]*4
    lib.sample_batch.restype=C.c_void_p
    lib.result_size.argtypes=[C.c_void_p,C.c_int];lib.result_size.restype=i
    lib.result_data.argtypes=[C.c_void_p,C.c_int];lib.result_data.restype=p
    lib.free_result.argtypes=[C.c_void_p];lib.free_result.restype=None
    return lib


class SharedTopology(Topology):
    def __init__(self, dataset='hi', cap=100):
        super().__init__(dataset,cap)
        self.lib=load_native()

    def batch(self, ids):
        ids = np.ascontiguousarray(ids, dtype=np.int64)
        assert len(ids) > 0 and ids.min() >= 0 and ids.max() < self.width
        handle = self.lib.sample_batch(*self.rawptr, self.width,
            ids.ctypes.data_as(C.POINTER(C.c_int64)), len(ids), self.nn, self.cap, *self.indexptr)
        if not handle:
            raise RuntimeError('Native daily graph construction failed')
        try:
            return tuple(copy_result(self.lib, handle, k) for k in range(8))
        finally:
            self.lib.free_result(handle)
'''


In [ ]:
# salt_daily/degree.c
SOURCES['salt_daily/degree.c'] = '''#include <stdint.h>
/* Original transaction counts; repeated/parallel transactions count separately. */
void prefix_counts(const int64_t *offsets, const uint32_t *ids,
                   const int32_t *minute, int64_t nodes, int64_t cutoff,
                   int64_t *result) {
    for (int64_t node = 0; node < nodes; ++node) {
        int64_t lo = offsets[node], hi = offsets[node + 1];
        while (lo < hi) {
            int64_t mid = lo + (hi - lo) / 2;
            if (minute[ids[mid]] < cutoff) lo = mid + 1;
            else hi = mid;
        }
        result[node] = lo - offsets[node];
    }
}

'''


In [ ]:
# salt_daily/extra_features.py
SOURCES['salt_daily/extra_features.py'] = '''"""Observed payment type and immutable first-counterparty port ranks.

Features depend on transaction IDs, never labels or membership in a mini-batch.
Parallel transactions remain distinct. The chronological ports are precomputed
by the preserved FraudGT implementation, with past ranks independent of future
transactions. Keep the raw values to match that implementation.
"""
from .config import PORTS, FEATURES as G
import numpy as np

FORMAT_NAMES = ('ACH', 'Bitcoin', 'Cash', 'Cheque', 'Credit Card', 'Reinvestment', 'Wire', 'Unknown')
PORT_ROOT = PORTS
INPUT_PATHS = (G/'fmt.npy', PORT_ROOT/'incoming_ports.npy', PORT_ROOT/'outgoing_ports.npy')


def encode_formats(codes):
    codes = np.asarray(codes, dtype=np.int64)
    codes = np.where((codes >= 0) & (codes < 7), codes, 7)
    return np.eye(len(FORMAT_NAMES), dtype=np.float32)[codes]


class ExtraEdgeFeatures:
    def __init__(self):
        self.formats, self.in_ports, self.out_ports = (
            np.load(path, mmap_mode='r') for path in INPUT_PATHS)
        assert len(self.formats) == len(self.in_ports) == len(self.out_ports)

    def values(self, ids):
        ids = np.asarray(ids, dtype=np.int64)
        ordered, restore = np.unique(ids, return_inverse=True)
        ports = np.column_stack((self.in_ports[ordered], self.out_ports[ordered]))
        assert np.all(ports >= 0), 'Uninitialized port feature'
        values = np.column_stack((encode_formats(self.formats[ordered]), ports)).astype(np.float32)
        return np.ascontiguousarray(values[restore])

    def append(self, original, ids):
        assert original.shape == (len(ids), 14)
        return np.ascontiguousarray(np.column_stack((original, self.values(ids))), dtype=np.float32)
'''


In [ ]:
# salt_daily/data.py
SOURCES['salt_daily/data.py'] = '''from .config import ROOT, LEGACY, GRAPH, FEATURES
from .storage import read_json, write_json, sha
import ctypes as C
import gc
import subprocess
import numpy as np
from runtime.graph_runtime import Topology, HIInputs
from .extra_features import ExtraEdgeFeatures
from .shared_topology import SharedTopology


def date(day):
    return str(np.datetime64('2022-08-01') + np.timedelta64(int(day), 'D'))


def cohort(scope):
    from .population import FullPopulation
    return FullPopulation(scope)


def daily_orders(data, epoch, shuffle=True, start_day_cursor=0):
    """Chronological days; shuffle targets within a day only."""
    if hasattr(data, 'orders'):
        yield from data.orders(epoch, shuffle, start_day_cursor)
        return
    for day in np.unique(data['day']):
        indices = np.flatnonzero(data['day'] == day)
        if shuffle:
            indices = np.random.default_rng(42000 + epoch * 100 + int(day)).permutation(indices)
        yield int(day), indices


class Degrees:
    def __init__(self, topology):
        self.topology = topology
        self.directory = ROOT / 'cache/degree'
        self.directory.mkdir(parents=True, exist_ok=True)
        source = ROOT / 'salt_daily/degree.c'
        library = self.directory / ('degree_' + sha(source)[:12] + '.so')
        if not library.exists():
            subprocess.run(['gcc', '-O3', '-std=c11', '-fPIC', '-shared', str(source), '-o', str(library)], check=True, capture_output=True)
        self.lib = C.CDLL(str(library))
        self.p64, self.p32, self.pu32 = C.POINTER(C.c_int64), C.POINTER(C.c_int32), C.POINTER(C.c_uint32)
        self.lib.prefix_counts.argtypes = [self.p64, self.pu32, self.p32, C.c_int64, C.c_int64, self.p64]
        self.lib.prefix_counts.restype = None
        self.loaded_day = None

    def load(self, day):
        if day == self.loaded_day:
            return self.counts
        path = self.directory / f'{date(day)}.npy'
        marker = path.with_suffix('.json')
        cutoff = (int(day) + 1) * 1440
        if not marker.exists():
            counts = np.empty((self.topology.nn, 2), dtype=np.int64)
            for direction, offset_index in enumerate((2, 6)):
                offsets, ids = self.topology.indices[offset_index:offset_index + 2]
                column = np.empty(self.topology.nn, dtype=np.int64)
                self.lib.prefix_counts(offsets.ctypes.data_as(self.p64), ids.ctypes.data_as(self.pu32),
                                       self.topology.minute.ctypes.data_as(self.p32), self.topology.nn,
                                       cutoff, column.ctypes.data_as(self.p64))
                counts[:, direction] = column
            temporary = path.with_suffix('.tmp.npy')
            np.save(temporary, counts)
            temporary.replace(path)
            write_json(marker, dict(day=int(day), cutoff_exclusive=cutoff, sha256=sha(path),
                                    degree_source='original incoming/outgoing transaction counts',
                                    source_sha256=sha(ROOT / 'salt_daily/degree.c')))
        metadata = read_json(marker)
        assert metadata['cutoff_exclusive'] == cutoff
        self.counts = np.load(path, mmap_mode='r')
        self.loaded_day = day
        return self.counts

    def calibration(self):
        # Fit only on the training graph, never Monitor/Val/Test. Daily contexts
        # remain capped at D; this global fitted scalar is a training transform.
        day = read_json(ROOT / 'protocol.json')['cohorts']['train_gnn']['max_day']
        counts = self.load(day)
        active = counts.sum(axis=1) > 0
        delta = np.log1p(counts[active]).mean(axis=0)
        assert np.all(np.isfinite(delta)) and np.all(delta > 0)
        return dict(delta=delta.tolist(), calibration_date=date(day), active_accounts=int(active.sum()),
                    original_transaction_count=int(counts[:, 0].sum()),
                    zero_degree_in_each_direction_included=True,
                    normalization='mean log1p original degree over active account union at training graph close',
                    fit_scope='training graph only; no Monitor/Val/Test', daily_graph_cutoff_unchanged=True)


class DailyData:
    def __init__(self, shared=True, device='cpu'):
        # The disjoint option is exclusively a validation/benchmark reference.
        protocol = read_json(ROOT/'protocol.json')
        backend = protocol['graph'].get('sampler_backend', 'cpu')
        if shared and backend == 'cuda' and device == 'cuda':
            import torch
            from .cuda_topology import CudaTopology
            torch.cuda.set_per_process_memory_fraction(protocol['training']['cuda_memory_fraction'])
            self.topology = CudaTopology('hi', cap=100)
        else:
            self.topology = (SharedTopology if shared else Topology)('hi', cap=100)
        self.inputs = HIInputs()
        self.extra_features = ExtraEdgeFeatures()
        self.degrees = Degrees(self.topology)
        contract = read_json(ROOT / 'docs/input_contract.json')
        excluded = {'amt_z_pay_ccy', 'pair_history_count', 'same_entity'}
        self.edge_indices = [i for i, name in enumerate(contract['edge_features']) if name not in excluded]
        assert len(self.edge_indices) == 14 and len(contract['node_features']) == 19
        self.gpu_features = None
        if shared and device == 'cuda' and protocol.get('input_pipeline', {}).get('backend') == 'cuda':
            assert backend == 'cuda', 'GPU inputs require the CUDA graph sampler'
            from .cuda_features import CudaFeatureStore
            self.gpu_features = CudaFeatureStore(self.inputs, self.extra_features,
                                                self.topology.width, self.edge_indices)
        self.day = None

    def set_day(self, day):
        self.day = int(day)
        self.inputs.set_day(date(day))
        self.degree = self.degrees.load(day)
        if self.gpu_features is not None:
            self.gpu_features.set_day(self.inputs.nodes, self.degree)

    def materialize(self, ids):
        """NumPy compatibility interface; training keeps the inputs on GPU."""
        if self.gpu_features is not None:
            arrays, graph = self.materialize_gpu(ids)
            return tuple(a.cpu().numpy() for a in arrays), tuple(a.cpu().numpy() for a in graph)
        return self.materialize_cpu(ids)

    def materialize_cpu(self, ids, graph=None):
        """Independent original feature path for CPU runs and exact validation."""
        ids = np.asarray(ids, dtype=np.int64)
        assert self.day is not None and len(ids) > 0
        assert np.all(self.topology.minute[ids] // 1440 == self.day), 'Mixed-day query batch'
        if graph is None:
            graph = self.topology.batch(ids)
        nodes, edges, sources, destinations, target_sources, target_destinations, flags, ego = graph
        assert np.all(self.topology.minute[edges] < (self.day + 1) * 1440), 'Future context edge'
        arrays = list(self.inputs.materialize(ids, graph))
        arrays[2] = np.ascontiguousarray(arrays[2][:, self.edge_indices])
        arrays[4] = np.ascontiguousarray(arrays[4][:, self.edge_indices])
        arrays[2] = self.extra_features.append(arrays[2], edges)
        arrays[4] = self.extra_features.append(arrays[4], ids)
        arrays.append(np.asarray(self.degree[nodes], dtype=np.float32))
        assert arrays[3].shape == (2, len(ids))
        assert np.array_equal(nodes[target_sources], self.topology.src[ids])
        assert np.array_equal(nodes[target_destinations], self.topology.dst[ids])
        for array in (arrays[0], arrays[2], arrays[4], arrays[7]):
            assert np.isfinite(array).all()
        return tuple(arrays), graph

    def materialize_gpu(self, ids, graph=None):
        import torch
        ids = np.asarray(ids, dtype=np.int64)
        assert self.day is not None and len(ids) > 0
        assert np.all(self.topology.minute[ids] // 1440 == self.day), 'Mixed-day query batch'
        if graph is None:
            graph = self.topology.batch_tensors(ids)
        targets = torch.tensor(ids, device=self.topology.device)
        nodes, edges, _, _, ts, td, _, _ = graph
        assert bool((self.topology.raw_gpu[2][edges] < (self.day+1)*1440).all()), 'Future context edge'
        assert torch.equal(nodes[ts], self.topology.raw_gpu[0][targets].long())
        assert torch.equal(nodes[td], self.topology.raw_gpu[1][targets].long())
        return self.gpu_features.materialize(targets, graph), graph

    def microbatches(self, data, indices, microbatch=8, max_edges=40000):
        def split(take):
            ids = data['ids'][take]
            graph = (self.topology.batch_tensors(ids) if self.gpu_features is not None
                     else self.topology.batch(ids))
            if len(graph[1]) > max_edges and len(take) > 1:
                middle = len(take) // 2
                del graph
                yield from split(take[:middle])
                yield from split(take[middle:])
            else:
                # A single large query is preserved, never dropped or truncated.
                materialize = self.materialize_gpu if self.gpu_features is not None else self.materialize_cpu
                arrays, graph = materialize(ids, graph)
                yield take, arrays, dict(nodes=len(graph[0]), edges=len(graph[1]))
                del arrays, graph
        for start in range(0, len(indices), microbatch):
            yield from split(indices[start:start + microbatch])


def tensors(arrays, device):
    import torch
    return tuple(torch.as_tensor(a, device=device,
                 dtype=torch.long if i in (1, 3) else torch.bool if i in (5, 6) else torch.float32)
                 for i, a in enumerate(arrays))


def cleanup():
    gc.collect()
    try:
        C.CDLL(None).malloc_trim(0)
    except AttributeError:
        pass

'''


In [ ]:
# salt_daily/shared_validation.py
SOURCES['salt_daily/shared_validation.py'] = '''"""Read-only verification helpers, not used by the training sampler."""
import numpy as np


def union_reference(disjoint,topology,ids):
    nodes=np.unique(disjoint[0])
    order=np.argsort(disjoint[1],kind='stable')
    ordered=disjoint[1][order]
    starts=np.r_[0,np.flatnonzero(ordered[1:]!=ordered[:-1])+1] if len(ordered) else np.empty(0,dtype=np.int64)
    edges=ordered[starts]
    flags=np.bitwise_or.reduceat(disjoint[6][order],starts) if len(starts) else np.empty(0,dtype=np.int64)
    ts=np.searchsorted(nodes,topology.src[ids]);td=np.searchsorted(nodes,topology.dst[ids])
    ego=np.zeros(len(nodes),np.int64);ego[ts]=1;ego[td]=1
    return (nodes,edges,np.searchsorted(nodes,topology.src[edges]),np.searchsorted(nodes,topology.dst[edges]),ts,td,flags,ego)


def check_union(disjoint,shared,topology,ids):
    expected=union_reference(disjoint,topology,ids)
    for actual,reference in zip(shared,expected):np.testing.assert_array_equal(actual,reference)
    return dict(targets=len(ids),disjoint_nodes=len(disjoint[0]),shared_nodes=len(shared[0]),
                disjoint_edges=len(disjoint[1]),shared_edges=len(shared[1]),
                exact_sampled_union=True,parallel_transactions_preserved=True,
                batch_ego_verified=True,target_mapping_verified=True)
'''


In [ ]:
# salt_daily/validate.py
SOURCES['salt_daily/validate.py'] = '''"""Population, loss, resume, metrics, and real daily-graph preflight."""
import io
import time
import unittest

import numpy as np
import torch

from .config import ROOT, GRAPH, FEATURES, DATA_ROOT
from .prepare import prepare
from .storage import read_json, write_json, sha, utc
from .data import DailyData, cohort, tensors, date, cleanup
from .model import DailySALT, model_kwargs
from .training import microbatch_loss, loss_denominator
from .extra_features import INPUT_PATHS
from .shared_validation import check_union
from runtime.graph_runtime import Topology
from .portable import check_inputs, code_manifest


def validate(device='cpu'):
    protocol = prepare()
    torch.set_num_threads(1)
    if device == 'cuda':
        torch.cuda.set_per_process_memory_fraction(protocol['training'].get('cuda_memory_fraction', .9))
    suite = unittest.defaultTestLoader.discover(str(ROOT/'tests'))
    output = io.StringIO()
    tests = unittest.TextTestRunner(stream=output, verbosity=2).run(suite)
    (ROOT/'unit_tests.log').write_text(output.getvalue())
    assert tests.wasSuccessful(), output.getvalue()
    loader = DailyData(device=device)
    reference_topology = Topology('hi', cap=100)
    calibration = read_json(ROOT/'calibration.json')
    assert calibration['calibration_date'] == '2022-09-25'
    records = []
    for scope, day in [('train_gnn', 7), ('train_gnn', 55), ('monitor', 56),
                       ('validation', 69), ('test', 74), ('test', 91)]:
        data = cohort(scope)
        span = next(s for s in data.meta['spans'] if s['day'] == day)
        labels = data['y'][span['start']:span['stop']]
        positions = np.array([np.flatnonzero(labels == 0)[0], np.flatnonzero(labels > 0)[0]]) + span['start']
        loader.set_day(day)
        begin = time.monotonic(); arrays, graph = loader.materialize(data['ids'][positions])
        assert arrays[2].shape[1] == arrays[4].shape[1] == 24
        original = loader.inputs.materialize(data['ids'][positions], graph)
        np.testing.assert_array_equal(arrays[0], original[0])
        np.testing.assert_array_equal(arrays[2][:, :14], original[2][:, loader.edge_indices])
        np.testing.assert_array_equal(arrays[4][:, :14], original[4][:, loader.edge_indices])
        for features, ids in ((arrays[2], graph[1]), (arrays[4], data['ids'][positions])):
            codes = np.asarray(loader.extra_features.formats[ids], dtype=np.int64)
            expected = np.where((codes >= 0) & (codes < 7), codes, 7)
            np.testing.assert_array_equal(features[:, 14:22].sum(1), np.ones(len(ids)))
            np.testing.assert_array_equal(features[:, 14:22].argmax(1), expected)
            np.testing.assert_array_equal(features[:, 22], loader.extra_features.in_ports[ids])
            np.testing.assert_array_equal(features[:, 23], loader.extra_features.out_ports[ids])
        assert np.all(loader.topology.minute[graph[1]] < (day+1)*1440)
        assert loader.inputs.day == date(day)
        for node in np.unique(graph[0])[:16]:
            for direction, offset_index in enumerate((2, 6)):
                offsets, ids = loader.topology.indices[offset_index:offset_index+2]
                incident = ids[offsets[node]:offsets[node+1]]
                assert loader.degree[node, direction] == np.count_nonzero(loader.topology.minute[incident] < (day+1)*1440)
        variants = {}
        rng = np.random.default_rng(4200+day)
        batch_positions = rng.choice(span['stop']-span['start'],
            size=min(protocol['training']['microbatch_max'], span['stop']-span['start']), replace=False)+span['start']
        # Include transaction endpoints from the highest-degree accounts where
        # the final incident transaction belongs to the current target day.
        extras = []
        for direction, offset_index in enumerate((2, 6)):
            offsets, incident_ids = loader.topology.indices[offset_index:offset_index+2]
            counts = np.asarray(loader.degree[:, direction])
            for node in np.argsort(counts)[-min(4, len(counts)):]:
                if counts[node] > 0:
                    tid = incident_ids[offsets[node]+counts[node]-1]
                    if loader.topology.minute[tid]//1440 == day:
                        candidate = np.searchsorted(data['ids'][span['start']:span['stop']], tid)+span['start']
                        if candidate < span['stop'] and data['ids'][candidate] == tid:
                            extras.append(candidate)
        batch_positions = np.unique(np.r_[batch_positions, extras]).astype(np.int64)
        check_ids = data['ids'][batch_positions[:protocol['training']['microbatch_max']]]
        independent_graph = reference_topology.batch(check_ids)
        shared_graph = loader.topology.batch(check_ids)
        sharing = check_union(independent_graph, shared_graph, reference_topology, check_ids)
        del independent_graph, shared_graph
        for variant in protocol['variants']:
            torch.manual_seed(42)
            model = DailySALT(variant, **model_kwargs(protocol, calibration['delta'])).to(device)
            logits = model(*tensors(arrays, device), return_logits=True)
            target = torch.as_tensor(data['y'][positions] > 0, dtype=torch.long, device=device)
            loss = microbatch_loss(logits, target, loss_denominator(data['y'][positions]))
            loss.backward()
            assert torch.isfinite(logits).all() and torch.isfinite(loss)
            assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)
            variants[variant] = dict(scored=2, finite_gradients=True)
            del logits, loss
            model.zero_grad(set_to_none=True)
            scored = 0; largest_batch = largest_edges = 0
            if device == 'cuda': torch.cuda.reset_peak_memory_stats()
            for take, large_arrays, metadata in loader.microbatches(data, batch_positions,
                    microbatch=protocol['training']['microbatch_max'],
                    max_edges=protocol['training']['max_edges_per_microbatch']):
                if loader.gpu_features is not None and variant == protocol['variants'][0]:
                    cpu_arrays, cpu_graph = loader.materialize_cpu(data['ids'][take])
                    for gpu_array, cpu_array in zip(large_arrays, cpu_arrays):
                        np.testing.assert_array_equal(gpu_array.cpu().numpy(), cpu_array)
                    del cpu_arrays, cpu_graph
                logits = model(*tensors(large_arrays, device), return_logits=True)
                target = torch.as_tensor(data['y'][take] > 0, dtype=torch.long, device=device)
                loss = microbatch_loss(logits, target, loss_denominator(data['y'][batch_positions]))
                assert torch.isfinite(logits).all() and torch.isfinite(loss)
                loss.backward(); scored += len(take)
                largest_batch = max(largest_batch, len(take)); largest_edges = max(largest_edges, metadata['edges'])
                del logits, loss, target, large_arrays
            assert scored == len(batch_positions)
            assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)
            peak = torch.cuda.max_memory_allocated() if device == 'cuda' else None
            if peak is not None:
                limit = torch.cuda.get_device_properties(0).total_memory * protocol['training']['cuda_memory_fraction']
                headroom = min(1024**3, .1*limit)
                assert peak < limit-headroom, 'Selected batch exceeds validated GPU headroom'
            variants[variant].update(enlarged_batch_scored=scored, largest_batch=largest_batch,
                                     largest_edges=largest_edges, cuda_peak_allocated_bytes=peak)
            del model; cleanup()
            if device == 'cuda': torch.cuda.empty_cache()
        records.append(dict(scope=scope, day=date(day), ids=data['ids'][positions].tolist(),
            normal_and_positive=True, nodes=len(graph[0]), edges=len(graph[1]), variants=variants,
            future_days_excluded=True, exact_original_degree_checked=True,
            base_features_unchanged=True, payment_and_port_values_verified=True,
            cuda_input_arrays_exact=loader.gpu_features is not None,
            high_degree_target_examples=len(extras), shared_graph_verification=sharing,
            seconds=time.monotonic()-begin))
        print('actual_day_pass', scope, date(day), flush=True)
        del arrays, graph; cleanup()
    # All retained Train/Monitor/Val targets must have features, not only old sampled targets.
    present = np.load(FEATURES/'present.npy', mmap_mode='r')
    minute = np.load(FEATURES/'minute.npy', mmap_mode='r')
    edge_present = loader.inputs.present
    missing = 0
    for start in range(0, len(minute), 1_000_000):
        values = minute[start:start+1_000_000]
        valid = (present[start:start+len(values)] != 0) & (values >= 7*1440) & (values < 74*1440)
        ids = np.flatnonzero(valid)+start
        missing += int(np.count_nonzero(edge_present[ids] == 0))
    assert missing == 0
    test_ids = cohort('test')['ids']
    assert np.all(loader.inputs.present[test_ids] != 0), 'Missing Test input features'
    stats = check_inputs()
    manifest = code_manifest()
    write_json(ROOT/'code_manifest.json', manifest)
    result = dict(status='pass', utc=utc(), device=device, unit_tests=tests.testsRun, unit_tests_skipped=len(tests.skipped), actual_days=records,
        code_manifest=manifest, protocol_sha256=sha(ROOT/'protocol.json'), input_file_stats=stats,
        all_population_counts={s: v['rows'] for s, v in protocol['cohorts'].items()},
        missing_train_monitor_validation_target_features=missing, test_feature_range_verified=True,
        production_training_performed=False, production_test_performed=False,
        test_labels_used_only_for_finite_gradient_smoke=True,
        test_smoke_only_no_optimizer_step_or_saved_test_model=True,
        test_smoke_targets_per_day={r['day']: {v: 2+x['enlarged_batch_scored']
            for v, x in r['variants'].items()} for r in records if r['scope'] == 'test'},
        limitation='Finite forward/backward and enlarged-batch checks only; full-population convergence not measured')
    write_json(ROOT/'preflight.json', result)
    return result
'''


In [ ]:
# salt_daily/cuda_topology.py
SOURCES['salt_daily/cuda_topology.py'] = '''"""Exact two-hop integer sampler using CUDA kernels compiled by NVRTC."""
import ctypes as C
import glob
import importlib.util
from pathlib import Path

import numpy as np
import torch
from .shared_topology import SharedTopology
from .config import ROOT


class Kernel:
    def __init__(self, source):
        spec = importlib.util.find_spec('nvidia.cuda_nvrtc')
        if spec is None:
            raise RuntimeError('CUDA graph sampling requires NVIDIA NVRTC (installed with CUDA PyTorch)')
        paths = glob.glob(str(Path(next(iter(spec.submodule_search_locations)))/'lib/libnvrtc.so*'))
        if not paths:
            raise RuntimeError('NVIDIA NVRTC shared library was not found')
        self.nvrtc=C.CDLL(paths[0]); self.driver=C.CDLL('libcuda.so.1')
        nv=self.nvrtc; dr=self.driver
        nv.nvrtcCreateProgram.argtypes=[C.POINTER(C.c_void_p),C.c_char_p,C.c_char_p,C.c_int,C.c_void_p,C.c_void_p]
        nv.nvrtcCompileProgram.argtypes=[C.c_void_p,C.c_int,C.POINTER(C.c_char_p)]
        for prefix in ('ProgramLog','PTX'):
            getattr(nv,'nvrtcGet'+prefix+'Size').argtypes=[C.c_void_p,C.POINTER(C.c_size_t)]
            getattr(nv,'nvrtcGet'+prefix).argtypes=[C.c_void_p,C.c_void_p]
        nv.nvrtcDestroyProgram.argtypes=[C.POINTER(C.c_void_p)]
        dr.cuModuleLoadData.argtypes=[C.POINTER(C.c_void_p),C.c_void_p]
        dr.cuModuleGetFunction.argtypes=[C.POINTER(C.c_void_p),C.c_void_p,C.c_char_p]
        dr.cuLaunchKernel.argtypes=[C.c_void_p]+[C.c_uint]*7+[C.c_void_p,C.c_void_p,C.c_void_p]
        program=C.c_void_p()
        self.check(nv.nvrtcCreateProgram(C.byref(program),source.encode(),b'salt_cuda.cu',0,None,None))
        major,minor=torch.cuda.get_device_capability()
        options=(C.c_char_p*2)(f'--gpu-architecture=compute_{major}{minor}'.encode(),b'--std=c++11')
        result=nv.nvrtcCompileProgram(program,2,options)
        size=C.c_size_t(); nv.nvrtcGetProgramLogSize(program,C.byref(size))
        message=C.create_string_buffer(size.value);nv.nvrtcGetProgramLog(program,message)
        if result: raise RuntimeError(message.value.decode())
        self.check(nv.nvrtcGetPTXSize(program,C.byref(size)))
        ptx=C.create_string_buffer(size.value);self.check(nv.nvrtcGetPTX(program,ptx))
        self.module=C.c_void_p();self.function=C.c_void_p()
        self.check(dr.cuModuleLoadData(C.byref(self.module),ptx))
        self.check(dr.cuModuleGetFunction(C.byref(self.function),self.module,b'expand'))
        self.check(nv.nvrtcDestroyProgram(C.byref(program)))

    @staticmethod
    def check(result):
        if result: raise RuntimeError(f'CUDA sampler API error {result}')

    def launch(self, states, day, cap, raw, indices, flags, counts=None, starts=None, next_states=None, mode=2):
        if not len(states): return
        def pointer(value): return C.c_void_p(value.data_ptr() if value is not None else 0)
        values=[pointer(states),C.c_longlong(len(states)),C.c_longlong(day),C.c_longlong(cap)]
        values += [pointer(v) for v in (*raw,*indices,flags,counts,starts,next_states)]
        values += [C.c_int(mode)]
        args=(C.c_void_p*len(values))(*[C.cast(C.byref(v),C.c_void_p) for v in values])
        self.check(self.driver.cuLaunchKernel(self.function,(len(states)*2+127)//128,1,1,128,1,1,0,
                    C.c_void_p(torch.cuda.current_stream().cuda_stream),args,None))


class CudaTopology(SharedTopology):
    def __init__(self,dataset='hi',cap=100,source=None):
        super().__init__(dataset,cap)
        self.attach(source)

    def attach(self,source=None):
        source = source or ROOT/'salt_daily/cuda_sampler.cu'
        if not torch.cuda.is_available(): raise RuntimeError('CUDA sampler requires CUDA')
        assert all(a.dtype == np.int32 and a.flags.c_contiguous for a in (self.src,self.dst,self.minute))
        assert self.width < 2**31, 'GPU CSR uses signed int32 edge IDs'
        self.device=torch.device('cuda',torch.cuda.current_device())
        self.raw_gpu=tuple(torch.tensor(a,device=self.device) for a in (self.src,self.dst,self.minute))
        self.indices_gpu=tuple(torch.tensor(a.view(np.int32) if a.dtype==np.uint32 else a,device=self.device) for a in self.indices)
        self.flags_gpu=torch.zeros(self.width,dtype=torch.int32,device=self.device)
        self.kernel=Kernel(Path(source).read_text())

    @torch.no_grad()
    def batch_tensors(self,ids):
        assert torch.cuda.current_device() == self.device.index, 'CUDA sampler device changed'
        ids=np.ascontiguousarray(ids,dtype=np.int64)
        assert len(ids)>0 and ids.min()>=0 and ids.max()<self.width
        minutes=self.minute[ids]
        assert (minutes>=0).all() and (minutes//1440==minutes[0]//1440).all(), 'Mixed-day query batch'
        day=int(minutes[0]//1440)
        targets=torch.tensor(ids,device=self.device)
        src,dst,minute=self.raw_gpu
        target_src=src[targets].long(); target_dst=dst[targets].long()
        offsets=minute[targets].long()%1440
        states=torch.unique(torch.cat((target_src*1440+offsets,target_dst*1440+offsets)))
        counts=torch.empty(len(states)*2,dtype=torch.int64,device=self.device)
        flags=self.flags_gpu; flags.zero_()
        self.kernel.launch(states,day,self.cap,self.raw_gpu,self.indices_gpu,flags,counts=counts,mode=0)
        ends=counts.cumsum(0); starts=ends-counts
        next_states=torch.empty(int(ends[-1]),dtype=torch.int64,device=self.device)
        self.kernel.launch(states,day,self.cap,self.raw_gpu,self.indices_gpu,flags,starts=starts,next_states=next_states,mode=1)
        frontier=torch.unique(next_states)
        self.kernel.launch(frontier,day,self.cap,self.raw_gpu,self.indices_gpu,flags,mode=2)
        del next_states,frontier,states,counts,ends,starts
        edges=flags.nonzero().flatten()
        edge_src=src[edges].long();edge_dst=dst[edges].long()
        nodes=torch.unique(torch.cat((edge_src,edge_dst,target_src,target_dst)))
        local_src=torch.searchsorted(nodes,edge_src);local_dst=torch.searchsorted(nodes,edge_dst)
        ts=torch.searchsorted(nodes,target_src);td=torch.searchsorted(nodes,target_dst)
        ego=torch.zeros(len(nodes),dtype=torch.int64,device=self.device);ego[ts]=1;ego[td]=1
        return nodes,edges,local_src,local_dst,ts,td,flags[edges].long(),ego

    def batch(self, ids):
        return tuple(v.cpu().numpy() for v in self.batch_tensors(ids))
'''


In [ ]:
# salt_daily/cuda_sampler.cu
SOURCES['salt_daily/cuda_sampler.cu'] = '''typedef long long I;
typedef unsigned int U;

__device__ I lower_edge(const int *ids, const int *minute, I lo, I hi, I target) {
    while (lo < hi) {
        I middle = lo + (hi-lo)/2;
        if (minute[ids[middle]] < target) lo=middle+1; else hi=middle;
    }
    return lo;
}

// mode 0 counts first-hop neighbors; 1 fills them; 2 only marks edges.
extern "C" __global__ void expand(
    const I *states, I count, I day, I cap, const int *src, const int *dst,
    const int *minute, const I *ipoff, const int *ip, const I *itoff, const int *it,
    const I *opoff, const int *op, const I *otoff, const int *ot,
    U *flags, I *counts, const I *starts, I *next, int mode) {
    I index = (I)blockIdx.x*blockDim.x+threadIdx.x;
    if (index >= count*2) return;
    int dir = index%2;
    I state=states[index/2], node=state/1440, offset=state%1440;
    I target=day*1440+offset, cutoff=(day+1)*1440;
    const I *poff=dir?opoff:ipoff, *toff=dir?otoff:itoff;
    const int *pi=dir?op:ip, *ti=dir?ot:it, *neighbor=dir?dst:src;
    I found=0, cursor=(mode==1)?starts[index]:0;
    for (I j=poff[node]; j<poff[node+1] && found<cap; ++j) {
        I edge=pi[j];
        if (minute[edge]>=cutoff || minute[edge]==target) continue;
        ++found;
        if (mode) atomicOr(flags+edge, dir?2U:1U);
        if (mode==1) next[cursor++]=(I)neighbor[edge]*1440+offset;
    }
    I lo=lower_edge(ti,minute,toff[node],toff[node+1],target);
    I hi=lower_edge(ti,minute,lo,toff[node+1],target+1);
    if (mode==0) {counts[index]=found+hi-lo; return;}
    for (I j=lo; j<hi; ++j) {
        I edge=ti[j];
        atomicOr(flags+edge, dir?2U:1U);
        if (mode==1) next[cursor++]=(I)neighbor[edge]*1440+offset;
    }
}
'''


In [ ]:
# salt_daily/cuda_features.py
SOURCES['salt_daily/cuda_features.py'] = '''"""Lossless resident input features: FP32 numeric values and packed categories."""
import time
import numpy as np
import torch


class CudaFeatureStore:
    def __init__(self, inputs, extra, width, edge_indices, device='cuda', block=500_000):
        assert edge_indices == [0,1,2,3,4,5,6,7,9,10,12,13,14,15]
        self.device = torch.device(device)
        self.numeric = torch.empty((width,12),dtype=torch.float32,device=device)
        self.categorical = torch.empty(width,dtype=torch.uint8,device=device)
        begin = time.monotonic()
        if width > 1_000_000:
            print(f'CUDA input cache loading: {width:,} transactions (no labels)',flush=True)
        for start in range(0,width,block):
            stop = min(width,start+block); ids = np.arange(start,stop,dtype=np.int64)
            present = ids < len(inputs.edge)
            present[present] = inputs.present[ids[present]] != 0
            valid = present
            selected = ids[valid]
            raw = inputs.edges(selected)
            booleans = raw[:,edge_indices[10:]]
            assert np.all((booleans == 0) | (booleans == 1)), 'Packed fields must be exactly binary'
            incoming = extra.in_ports[selected]; outgoing = extra.out_ports[selected]
            # Missing port entries retain an invalid marker and fail if selected.
            ports_valid = (incoming >= 0) & (outgoing >= 0)
            values = np.zeros((len(ids),12),dtype=np.float32)
            values[valid,:10] = raw[:,edge_indices[:10]]
            values[valid,10] = incoming; values[valid,11] = outgoing
            codes = extra.formats[selected].astype(np.int64)
            codes = np.where((codes >= 0) & (codes < 7),codes,7).astype(np.uint8)
            packed = np.zeros(len(ids),dtype=np.uint8)
            bits = sum(booleans[:,i].astype(np.uint8) << i for i in range(4))
            packed[valid] = (ports_valid.astype(np.uint8) << 7) | (codes << 4) | bits
            self.numeric[start:stop].copy_(torch.from_numpy(values))
            self.categorical[start:stop].copy_(torch.from_numpy(packed))
            if width > 1_000_000 and (stop % 40_000_000 == 0 or stop == width):
                print(f'CUDA input cache: {stop:,}/{width:,}',flush=True)
        torch.cuda.synchronize()
        self.initialization_seconds = time.monotonic()-begin
        self.nodes = self.degree = None

    def set_day(self, nodes, degree):
        self.nodes = torch.tensor(nodes,dtype=torch.float32,device=self.device)
        self.degree = torch.tensor(degree,dtype=torch.float32,device=self.device)

    @torch.no_grad()
    def edges(self, ids):
        packed = self.categorical[ids].long()
        assert bool(((packed & 128) != 0).all()), 'Missing input features or port ranks'
        values = self.numeric[ids]
        output = torch.empty((len(ids),24),dtype=torch.float32,device=self.device)
        output[:,:10] = values[:,:10]
        for i in range(4):
            output[:,10+i] = (packed >> i) & 1
        output[:,14:22] = 0
        output[:,14:22].scatter_(1,((packed >> 4) & 7)[:,None],1.)
        output[:,22:24] = values[:,10:12]
        return output

    @torch.no_grad()
    def materialize(self, ids, graph):
        nodes,edges,src,dst,ts,td,flags,ego = graph
        x = torch.empty((len(nodes),20),dtype=torch.float32,device=self.device)
        x[:,:19] = self.nodes[nodes]; x[:,19] = ego
        arrays = (x,torch.stack((src,dst)),self.edges(edges),torch.stack((ts,td)),
                  self.edges(ids),(flags&1)>0,(flags&2)>0,self.degree[nodes])
        for i in (0,2,4,7):
            assert bool(torch.isfinite(arrays[i]).all()), 'Nonfinite GPU input'
        return arrays
'''


In [ ]:
RUN_GRAPH_INDEX = False
if RUN_GRAPH_INDEX:
    command("graph")


In [ ]:
RUN_PREPARE_INPUTS = False
if RUN_PREPARE_INPUTS:
    command("prepare")


In [ ]:
RUN_VALIDATE_INPUTS = False
if RUN_VALIDATE_INPUTS:
    command("validate")


In [ ]:
path = WORK_DIR / "protocol.json"
if path.exists():
    protocol = json.loads(path.read_text())
    for scope, item in protocol["cohorts"].items():
        print(scope, f"{item['rows']:,}건", "양성", f"{item['positive']:,}건")
    result = WORK_DIR / "preflight.json"
    print("입력 검증:", json.loads(result.read_text())["status"] if result.exists() else "아직 실행 전")
else:
    print("그래프 생성 및 학습 입력 준비를 먼저 실행하세요.")
